In [ ]:
-- ********************************************************************--
-- author:梁学义
-- create time:2025-09-01 20:38:16
-- ********************************************************************--
DROP TABLE IF EXISTS monthly_report_order_driver;

--===========================================
--订单表driver，加上api-app,app-app,app复贷标签
--===========================================
CREATE TABLE monthly_report_order_driver AS

SELECT  x.*
       ,CASE WHEN y.order_number IS NOT NULL THEN 'API_APP'
             WHEN z.app_app_order_number IS NOT NULL THEN 'APP_APP'  ELSE 'APP复贷' END AS loan_flag
FROM
(
	SELECT  first_order_number
	       ,order_number
	       ,user_no
	       ,cust_no
	       ,period
	       ,loan_amt
	       ,loan_time
	       ,first_order_time
           ,asset_type_flag
	       ,fee_rate    --0.3599 
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE 1 = 1
	AND pt = '${bizdate}'
	AND app IN ('xyf01', 'fxk') --xyf01和fxk贷款 
	AND business_line IN ('APP', '小程序端') --定义APP贷款 
	AND loan_status = 'success'
	AND loan_flag IN ('复贷', '加贷')  --复贷 
	AND loan_time IS NOT NULL
	AND DATE(loan_time) >= '2024-11-01'
	AND DATE(loan_time) <= '2025-11-20'   --每次更新一下日期条件
) x  
LEFT JOIN --api-app成交订单 
(
	SELECT  DISTINCT order_number
	FROM xyf_bi.wzq_order_table
	WHERE 复贷客群分组 = 'API拉回APP' 
) y
ON x.order_number = y.order_number
LEFT JOIN --app-app成交订单 
(
	SELECT  a.order_number AS shoudai_order_number
	       ,b.order_number AS app_app_order_number
	FROM xyf_dws.dws_inloan_user_order_df a
	INNER JOIN xyf_dws.dws_inloan_user_order_df b
	ON a.pt = '${bizdate}' AND b.pt = '${bizdate}'
    AND a.user_no = b.user_no 
    AND a.loan_status = 'success' AND b.loan_status = 'success' 
    AND a.business_line IN ('APP', '小程序端') AND b.business_line IN ('APP', '小程序端') 
    AND a.loan_flag IN ('首贷') AND b.loan_flag IN ('复贷', '加贷') 
    QUALIFY ROW_NUMBER() OVER(PARTITION BY a.order_number ORDER BY b.loan_time) = 1
) z
ON x.order_number = z.app_app_order_number;
--===============
--1.大盘放款及完成度
--===============
SELECT  substr(loan_time,1,7) AS loan_mth
       ,loan_flag
       ,SUM(loan_amt)         AS loan_amt
FROM monthly_report_order_driver
GROUP BY  substr(loan_time,1,7)
         ,loan_flag
ORDER BY  substr(loan_time,1,7)
         ,loan_flag;

--===============
--2.vintage 无额外放开版
--===============

SELECT  substr(a.loan_time,1,7)                                                                   AS loan_mth
       ,COUNT(a.order_number)                                                                     AS order_cnt
       ,SUM(a.loan_amt)                                                                           AS loan_amt
       ,SUM(CASE WHEN y0_1_30 = 1 AND y1_1_30 = 1 THEN y3_1_30 ELSE 0 END)/100                    AS mob_1_30_dpd
       ,SUM(CASE WHEN y0_2_30 = 1 AND y1_2_30 = 1 THEN y3_2_30 ELSE 0 END)/100                    AS mob_2_30_dpd
       ,SUM(CASE WHEN y0_3_30 = 1 AND y1_3_30 = 1 THEN y3_3_30 ELSE 0 END)/100                    AS mob_3_30_dpd
       ,SUM(CASE WHEN y0_4_30 = 1 AND y1_4_30 = 1 THEN y3_4_30 ELSE 0 END)/100                    AS mob_4_30_dpd
       ,SUM(CASE WHEN y0_5_30 = 1 AND y1_5_30 = 1 THEN y3_5_30 ELSE 0 END)/100                    AS mob_5_30_dpd
       ,SUM(CASE WHEN y0_6_30 = 1 AND y1_6_30 = 1 THEN y3_6_30 ELSE 0 END)/100                    AS mob_6_30_dpd
       ,SUM(CASE WHEN y0_7_30 = 1 AND y1_7_30 = 1 THEN y3_7_30 ELSE 0 END)/100                    AS mob_7_30_dpd
       ,SUM(CASE WHEN y0_8_30 = 1 AND y1_8_30 = 1 THEN y3_8_30 ELSE 0 END)/100                    AS mob_8_30_dpd
       ,SUM(CASE WHEN y0_9_30 = 1 AND y1_9_30 = 1 THEN y3_9_30 ELSE 0 END)/100                    AS mob_9_30_dpd
       ,SUM(CASE WHEN y0_10_30 = 1 AND y1_10_30 = 1 THEN y3_10_30 ELSE 0 END)/100                 AS mob_10_30_dpd
       ,SUM(CASE WHEN y0_11_30 = 1 AND y1_11_30 = 1 THEN y3_11_30 ELSE 0 END)/100                 AS mob_11_30_dpd
       ,SUM(CASE WHEN y0_12_30 = 1 AND y1_12_30 = 1 THEN y3_12_30 ELSE 0 END)/100                 AS mob_12_30_dpd

       ,SUM(CASE WHEN y0_1_30 = 1 AND y1_1_30 = 1 THEN y3_1_30 ELSE 0 END)/100/SUM(a.loan_amt)    AS mob_1_30_dpd_rate
       ,SUM(CASE WHEN y0_2_30 = 1 AND y1_2_30 = 1 THEN y3_2_30 ELSE 0 END)/100/SUM(a.loan_amt)    AS mob_2_30_dpd_rate
       ,SUM(CASE WHEN y0_3_30 = 1 AND y1_3_30 = 1 THEN y3_3_30 ELSE 0 END)/100/SUM(a.loan_amt)    AS mob_3_30_dpd_rate
       ,SUM(CASE WHEN y0_4_30 = 1 AND y1_4_30 = 1 THEN y3_4_30 ELSE 0 END)/100/SUM(a.loan_amt)    AS mob_4_30_dpd_rate
       ,SUM(CASE WHEN y0_5_30 = 1 AND y1_5_30 = 1 THEN y3_5_30 ELSE 0 END)/100/SUM(a.loan_amt)    AS mob_5_30_dpd_rate
       ,SUM(CASE WHEN y0_6_30 = 1 AND y1_6_30 = 1 THEN y3_6_30 ELSE 0 END)/100/SUM(a.loan_amt)    AS mob_6_30_dpd_rate
       ,SUM(CASE WHEN y0_7_30 = 1 AND y1_7_30 = 1 THEN y3_7_30 ELSE 0 END)/100/SUM(a.loan_amt)    AS mob_7_30_dpd_rate
       ,SUM(CASE WHEN y0_8_30 = 1 AND y1_8_30 = 1 THEN y3_8_30 ELSE 0 END)/100/SUM(a.loan_amt)    AS mob_8_30_dpd_rate
       ,SUM(CASE WHEN y0_9_30 = 1 AND y1_9_30 = 1 THEN y3_9_30 ELSE 0 END)/100/SUM(a.loan_amt)    AS mob_9_30_dpd_rate
       ,SUM(CASE WHEN y0_10_30 = 1 AND y1_10_30 = 1 THEN y3_10_30 ELSE 0 END)/100/SUM(a.loan_amt) AS mob_10_30_dpd_rate
       ,SUM(CASE WHEN y0_11_30 = 1 AND y1_11_30 = 1 THEN y3_11_30 ELSE 0 END)/100/SUM(a.loan_amt) AS mob_11_30_dpd_rate
       ,SUM(CASE WHEN y0_12_30 = 1 AND y1_12_30 = 1 THEN y3_12_30 ELSE 0 END)/100/SUM(a.loan_amt) AS mob_12_30_dpd_rate

       ,SUM(CASE WHEN y0_1_30 = 1 AND y1_1_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)        AS mob_1_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_2_30 = 1 AND y1_2_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)        AS mob_2_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_3_30 = 1 AND y1_3_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)        AS mob_3_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_4_30 = 1 AND y1_4_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)        AS mob_4_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_5_30 = 1 AND y1_5_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)        AS mob_5_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_6_30 = 1 AND y1_6_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)        AS mob_6_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_7_30 = 1 AND y1_7_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)        AS mob_7_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_8_30 = 1 AND y1_8_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)        AS mob_8_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_9_30 = 1 AND y1_9_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)        AS mob_9_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_10_30 = 1 AND y1_10_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)      AS mob_10_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_11_30 = 1 AND y1_11_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)      AS mob_11_30_dpd_cnt_rate
       ,SUM(CASE WHEN y0_12_30 = 1 AND y1_12_30 = 1 THEN 1 ELSE 0 END)/COUNT(a.order_number)      AS mob_12_30_dpd_cnt_rate
FROM xyf_dws.dws_inloan_user_order_df a
INNER JOIN xyf_dws.dws_repay_risk_order_bill_mob_df b
ON a.pt = '${bizdate}'
AND b.pt = '${bizdate}' 
AND DATE(a.loan_time) >= '2024-07-01' 
AND a.order_number = b.order_number 
AND a.app IN ('xyf01', 'fxk') 
AND lower(a.app) = lower(a.inner_app) 
AND a.loan_flag IN ('复贷', '加贷') 
AND a.loan_status = 'success'
LEFT JOIN
(
	SELECT  DISTINCT order_number
	FROM xyf_fengkong_dev.wzq_order_b5_b4_compare_base
	WHERE 老客客群分组 = 'api拉回app'
	AND 边界分组 RLIKE 'VIP' 
	UNION ALL
	SELECT  order_number
	FROM xyf_fengkong_dev.zsh_tek_fk_ord_flg
	WHERE tek_fk_ord_flg = '额外放开' 
	UNION ALL
	SELECT  order_number
	FROM xyf_fengkong_dev.zsh_vip_fk_ord_flg
	WHERE vip_fk_ord_flg = '额外放开' 
)c
ON a.order_number = c.order_number
WHERE c.order_number IS NULL
GROUP BY  substr(a.loan_time,1,7)
ORDER BY  substr(a.loan_time,1,7) ASC;

--===============
--3.1 5.1 放款期限和放款价格
--===============

SELECT  substr(loan_time,1,7)                    AS loan_mth
       ,AVG(period)                              AS avg_period
       ,SUM(loan_amt*period)/SUM(loan_amt)       AS avg_period_amt_weighted
       ,AVG(fee_rate)*100                        AS avg_price
       ,SUM(loan_amt*fee_rate)/SUM(loan_amt)*100 AS avg_price_amt_weighted
FROM monthly_report_order_driver
GROUP BY  substr(loan_time,1,7)
ORDER BY  substr(loan_time,1,7);

--===============
--3.2 不同期数放款量
--===============
SELECT  substr(loan_time,1,7) AS loan_mth
       ,period
       ,SUM(loan_amt)         AS loan_amt
FROM monthly_report_order_driver
GROUP BY  substr(loan_time,1,7)
         ,period
ORDER BY  substr(loan_time,1,7)
         ,period;

--===============
--4 24价格放款数量, A1 A2放款
--===============
SELECT  substr(loan_time,1,7)        AS loan_mth
       ,asset_type_flag
       ,COUNT(DISTINCT order_number) AS loan_cnt
       ,SUM(loan_amt)                AS loan_amt
FROM monthly_report_order_driver
GROUP BY  substr(loan_time,1,7)
         ,asset_type_flag
ORDER BY  substr(loan_time,1,7)
         ,asset_type_flag;

--A1,A2
SELECT  substr(loan_time,1,7)        AS loan_mth
       ,apply_24_risk_qualification
       ,COUNT(DISTINCT order_number) AS loan_cnt
       ,SUM(loan_amt)                AS loan_amt
FROM
(
	SELECT  DISTINCT a.*
	       ,CASE WHEN b.model_value <= 1.2 THEN 'qualified'  ELSE 'unqualified' END AS apply_24_risk_qualification
	FROM monthly_report_order_driver a
	LEFT JOIN xyf_dwd.dwd_risk_model_b_card_df b
	ON a.cust_no = b.cust_no AND b.pt >= '20240501' AND DATE(a.loan_time) = DATE(b.decision_time)
)
GROUP BY  substr(loan_time,1,7)
         ,apply_24_risk_qualification
ORDER BY  substr(loan_time,1,7)
         ,apply_24_risk_qualification;

--========
--4.2 资产类别 app复贷 风险24， 24+权益，36
--========
WITH orders_updated AS
(
	SELECT  orders.*
	       ,CASE WHEN ori_price_overwritten.ori_order_number IS NOT NULL THEN 'I36'
	             WHEN application.ori_order_number IS NOT NULL THEN 'I36'
	             WHEN orders.asset_type_flag = 'I36' THEN 'I36'  ELSE 'I24' END AS ori_risk_price
	FROM
	(
		SELECT  first_order_number
		       ,loan_time
		       ,loan_amt
		       ,asset_type_flag
		FROM xyf_dws.dws_inloan_user_order_df order_info
		WHERE pt = '${bizdate}'
		AND loan_status = 'success'
		AND loan_flag IN ('复贷','加贷')
		AND business_line IN ('APP', '小程序端')
		AND DATE(loan_time) >= '2024-10-01'
	) orders
	LEFT JOIN xyf_fengkong_dev.scq_fy_history_risk_price_modify ori_price_overwritten
	ON orders.first_order_number = ori_price_overwritten.ori_order_number
	LEFT JOIN
	(
		SELECT  ori_order_number
		       ,ori_risk_price
		       ,risk_price
		FROM xyf_dwd.dwd_inloan_loan_apply_main_df
		WHERE 1 = 1
		AND pt = '${bizdate}'
		AND risk_price_type = 'I'
		AND ori_risk_price = 0.36
	) application
	ON orders.first_order_number = application.ori_order_number
)
SELECT  substr(loan_time,1,7) loan_mth
       ,ori_risk_price
       ,asset_type_flag
       ,SUM(loan_amt) AS loan_amt
FROM orders_updated
GROUP BY  substr(loan_time,1,7)
         ,ori_risk_price
         ,asset_type_flag
ORDER BY  substr(loan_time,1,7)
         ,ori_risk_price
         ,asset_type_flag
;


--==============================================================================================
-- 5.2 APR 口径价格 
--==============================================================================================
--==interest_fee
WITH apr AS
(
	SELECT  order_number
	       ,SUM(initial_principal)                                                                 AS initial_principal
	       ,SUM(nvl(initial_interest,0)+nvl(initial_after_loan_fee,0)+nvl(initial_platform_fee,0)) AS initial_interest_fee
	FROM xyf_dwd.dwd_repay_loan_repay_plan_df
	WHERE pt = '${bizdate}'
	GROUP BY  order_number
), 

order_apr AS(
SELECT  a.cust_no
       ,a.user_no AS app_user_id
       ,a.loan_time
       ,a.loan_amt
       ,b.initial_principal
       ,b.initial_interest_fee
FROM xyf_dws.dws_inloan_user_order_df a
INNER JOIN apr b
ON a.order_number = b.order_number AND a.pt = '${bizdate}' 
AND DATE(a.loan_time) >= '2024-10-01' AND a.app IN ('xyf01', 'fxk') AND a.loan_status = 'success' 
AND a.loan_flag IN ('复贷','加贷') AND a.business_line IN ('APP', '小程序端') AND a.loan_time IS NOT NULL 
)

SELECT  substr(loan_time,1,7)     AS loan_mth
       ,SUM(loan_amt)             AS loan_amt
       ,SUM(initial_principal)    AS principal
       ,SUM(initial_interest_fee) AS fee
FROM order_apr
GROUP BY  substr(loan_time,1,7)
ORDER BY  substr(loan_time,1,7);

--===============复贷会员卡，提额卡收入
WITH vip AS
(
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,pay_time        AS tran_time
	       ,real_card_price AS pay_amt
	       ,0               AS refund_amt
	       ,'leap'          AS card_type
	FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
	WHERE pt = '${bizdate}'
	AND DATE(order_time) >= '2025-05-24'
	AND pay_time IS NOT NULL 
	UNION ALL
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,act_refund_time AS tran_time
	       ,0               AS pay_amt
	       ,refund_amount   AS refund_amt
	       ,'leap'          AS card_type
	FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
	WHERE pt = '${bizdate}'
	AND DATE(order_time) >= '2025-05-24'
	AND act_refund_time IS NOT NULL 
	UNION ALL
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,pay_time            AS tran_time
	       ,real_card_price/100 AS pay_amt
	       ,0                   AS refund_amt
	       ,'vip'               AS card_type
	FROM xyf_dwd.dwd_user_vip_order_df
	WHERE pt = '${bizdate}'
	AND vip_card_type = 1
	AND if_validation <> 0
	AND pay_time IS NOT NULL 
	UNION ALL
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,act_refund_time   AS tran_time
	       ,0                 AS pay_amt
	       ,refund_amount/100 AS refund_amt
	       ,'vip'             AS card_type
	FROM xyf_dwd.dwd_user_vip_order_df
	WHERE pt = '${bizdate}'
	AND vip_card_type = 1
	AND if_validation <> 0
	AND act_refund_time IS NOT NULL 
), 
tek AS(
SELECT  app_user_id
       ,cust_no
       ,order_time
       ,order_time       AS tran_time
       ,real_order_price AS pay_amt
       ,0                AS refund_amt
       ,'tek'            AS card_type
       ,'复贷'             AS vip_classifier
FROM xyf_dwd.dwd_user_tek_order_df
WHERE pt = '${bizdate}'
UNION ALL
SELECT  app_user_id
       ,cust_no
       ,order_time
       ,act_refund_time AS tran_time
       ,0               AS pay_amt
       ,refund_amount   AS refund_amt
       ,'tek'           AS card_type
       ,'复贷'            AS vip_classifier
FROM xyf_dwd.dwd_user_tek_order_df
WHERE pt = '${bizdate}'
AND act_refund_time IS NOT NULL 
), 

--app首贷
--商业化订单order早于app首贷放款的都算首贷，其余算作app复贷收入。
orders AS(
SELECT  cust_no
       ,loan_time
FROM xyf_dws.dws_inloan_user_order_df
WHERE pt = '${bizdate}'
AND app IN ('xyf01')
AND business_line IN ('APP', '小程序端') --定义APP贷款 
AND loan_status = 'success'
AND loan_flag = '首贷' 
QUALIFY ROW_NUMBER() OVER(PARTITION BY cust_no ORDER BY loan_time DESC) = 1 
), 

vip_classify AS(
SELECT  a.*
       ,CASE WHEN b.cust_no IS NOT NULL THEN '首贷'  ELSE '复贷' END AS vip_classifier
FROM vip a
LEFT JOIN orders b
ON a.cust_no = b.cust_no AND a.order_time < b.loan_time
UNION ALL
SELECT  *
FROM tek 
)

SELECT  substr(tran_time,1,7)
       ,card_type
       ,SUM(pay_amt)    AS pay_amt
       ,SUM(refund_amt) AS refund_amt
FROM vip_classify
WHERE DATE(tran_time) >= '2024-10-01'
AND vip_classifier = '复贷'
GROUP BY  substr(tran_time,1,7)
         ,vip_classifier
         ,card_type;

--===============
--7 额外放开
--===============
WITH ind_extra AS
(
	SELECT  DISTINCT order_number
	       ,`复贷客群分组`
	FROM xyf_fengkong_dev.hhn_fk_ord_flg_all
	WHERE 复贷客群分组 IN ('提额卡', '会员卡', '飞跃会员卡') 
)
SELECT  substr(loan_time,1,7)                                                                         AS loan_mth
       ,CASE WHEN b.order_number IS NOT NULL AND b.`复贷客群分组` = '提额卡' THEN '提额卡放开'
             WHEN b.order_number IS NOT NULL AND b.`复贷客群分组` = '会员卡' THEN '飞享放开'
             WHEN b.order_number IS NOT NULL AND b.`复贷客群分组` = '飞跃会员卡' THEN '飞跃放开至36+'  ELSE '无放开' END AS ind_extra
       ,COUNT(1)                                                                                      AS cnt
       ,COUNT(DISTINCT a.order_number)                                                                AS cnt_ord
       ,SUM(loan_amt)                                                                                 AS sum_amt
FROM monthly_report_order_driver a
LEFT JOIN ind_extra b
ON a.order_number = b.order_number
GROUP BY  substr(loan_time,1,7)
         ,ind_extra;
--===============
--8.1月内放款人数及人均
--===============
SELECT  substr(loan_time,1,7)                 AS loan_mth
       ,COUNT(DISTINCT user_no)               AS user_cnt
       ,SUM(loan_amt)/COUNT(DISTINCT user_no) AS user_monthly_amt
FROM monthly_report_order_driver
GROUP BY  substr(loan_time,1,7)
ORDER BY  substr(loan_time,1,7);
--===============
--8.2月内放款笔均
--===============
SELECT  substr(loan_time,1,7) AS loan_mth
       ,loan_flag
       ,AVG(loan_amt)         AS avg_order_amount
FROM monthly_report_order_driver
GROUP BY  substr(loan_time,1,7)
         ,loan_flag

UNION ALL

SELECT  substr(loan_time,1,7) AS loan_mth
       ,'总体'
       ,AVG(loan_amt)         AS avg_order_amount
FROM monthly_report_order_driver
GROUP BY  substr(loan_time,1,7);

--===============
--9. 在贷
--===============
WITH user_no_level_mth_perf AS
(
	SELECT  pt
	       ,user_no
	       ,SUM(CASE WHEN overdue_days <= 30 THEN loan_balance END)                                 AS loan_balance_30_minus
	       ,COUNT(DISTINCT CASE WHEN overdue_days <= 30 AND loan_balance > 0 THEN order_number END) AS loan_cnt_30_minus
	FROM xyf_dws.dws_repay_user_order_df
	WHERE pt IN ( SELECT DISTINCT day_id FROM xyf_dim.dim_pub_date WHERE day_id = day_monthend) OR pt = '20251120'
	GROUP BY  pt
	         ,user_no
	HAVING loan_cnt_30_minus > 0
)

SELECT  by_pt_user_no_level_loan_cnt.*
       ,by_pt.avg_loan_amt AS by_pt_avg_loan_amt
FROM
(
	SELECT  pt
	       ,CASE WHEN loan_cnt_30_minus = 1 THEN '在贷=1'  ELSE '在贷>=2' END AS user_no_level_loan_cnt
	       ,COUNT(DISTINCT user_no)                                      AS user_no_cnt
	       ,SUM(loan_balance_30_minus)/COUNT(DISTINCT user_no)           AS avg_loan_amt
	FROM user_no_level_mth_perf
	GROUP BY  pt
	         ,CASE WHEN loan_cnt_30_minus = 1 THEN '在贷=1'  ELSE '在贷>=2' END
) by_pt_user_no_level_loan_cnt
INNER JOIN
(
	SELECT  pt
	       ,SUM(loan_balance_30_minus)/COUNT(DISTINCT user_no) AS avg_loan_amt
	FROM user_no_level_mth_perf
	GROUP BY  pt
) by_pt
ON by_pt_user_no_level_loan_cnt.pt = by_pt.pt;


--===============
--10 登录人次 & 登录人群可发标占比
--===============
WITH login AS
(
	SELECT  substr(last_login_time,1,7)                          AS login_mth
	       ,b_card_model
	       ,user_no
	       ,regulation_reason
	       ,CASE WHEN b_card_model IS NULL OR b_card_model < 0 THEN '空'
	             WHEN b_card_model < 3 THEN 'AB'
	             WHEN b_card_model < 5 THEN 'CD'
	             WHEN b_card_model < 7 THEN 'EF'
	             WHEN b_card_model < 9 THEN 'GH'  ELSE 'IJK' END AS b_card_category
	FROM xyf_ads.ads_user_market_portfolio_label_df
	WHERE pt >= '20240501' 
    QUALIFY ROW_NUMBER() OVER(PARTITION BY DATE(last_login_time), user_no ORDER BY pt) = 1
)
SELECT  login_mth
       ,b_card_category
       ,COUNT(user_no)                                                 AS login_instance
       ,COUNT(CASE WHEN regulation_reason IN ('不管制') THEN user_no END) AS line_available_login_instance
FROM login
WHERE login_mth >= '2024-10'
GROUP BY  login_mth
         ,b_card_category
ORDER BY  login_mth
         ,b_card_category;

--========================
--11 12 发起 & 资金通过率
--========================
WITH apply_risk_score AS
(
	SELECT  a.*
	       ,CASE WHEN b.model_value IS NULL OR b.model_value < 0 THEN '空'
	             WHEN b.model_value < 3 THEN 'AB'
	             WHEN b.model_value < 5 THEN 'CD'
	             WHEN b.model_value < 7 THEN 'EF'
	             WHEN b.model_value < 9 THEN 'GH'  ELSE 'IJK' END AS apply_model_score_category
	FROM
	(
		SELECT  first_order_number
		       ,first_order_time
		       ,risk_status
		       ,loan_status
		       ,loan_time
		       ,cust_no
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE 1 = 1
		AND pt = '${bizdate}'
		AND app IN ('xyf01', 'fxk')
		AND business_line IN ('APP', '小程序端')
		AND loan_flag IN ('复贷', '加贷')
		AND DATE(first_order_time) >= '2024-11-01'
		AND DATE(first_order_time) <= '2025-11-20'
	)a --每次更新一下日期条件 
	LEFT JOIN xyf_dwd.dwd_risk_model_b_card_df b
	ON a.cust_no = b.cust_no AND b.pt >= '20240901' AND DATE(a.first_order_time) = DATE(b.decision_time)
)

SELECT  substr(first_order_time,1,7)                                                                                  AS apply_month
       ,apply_model_score_category
       ,COUNT(DISTINCT first_order_number)                                                                            AS order_cnt
       ,COUNT(DISTINCT CASE WHEN risk_status = 'pass' THEN first_order_number END)                                    AS risk_pass_order_cnt
       ,COUNT(DISTINCT CASE WHEN risk_status = 'pass' THEN first_order_number END)/COUNT(DISTINCT first_order_number) AS risk_pass_rate
       ,COUNT(DISTINCT CASE WHEN loan_status = 'success' THEN first_order_number END)/COUNT(DISTINCT CASE WHEN risk_status = 'pass' THEN first_order_number END) AS fund_pass_rate
       ,COUNT(DISTINCT CASE WHEN loan_status = 'success' AND (unix_timestamp(loan_time) - unix_timestamp(first_order_time)) BETWEEN 0 AND 3600*2 THEN first_order_number END)/COUNT(DISTINCT CASE WHEN risk_status = 'pass' THEN first_order_number END) AS fund_2h_pass_rate
FROM apply_risk_score
GROUP BY  substr(first_order_time,1,7)
         ,apply_model_score_category;

--========================
--13 放款人次
--========================
WITH loan_risk_score AS
(
	SELECT  a.*
	       ,CASE WHEN b.model_value IS NULL OR b.model_value < 0 THEN '空'
	             WHEN b.model_value < 3 THEN 'AB'
	             WHEN b.model_value < 5 THEN 'CD'
	             WHEN b.model_value < 7 THEN 'EF'
	             WHEN b.model_value < 9 THEN 'GH'  ELSE 'IJK' END AS loan_model_score_category
	FROM
	(
		SELECT  first_order_number
		       ,first_order_time
		       ,risk_status
		       ,loan_status
		       ,loan_time
		       ,cust_no
		       ,loan_amt
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE 1 = 1
		AND pt = '${bizdate}'
		AND app IN ('xyf01', 'fxk')
		AND lower(app) = lower(inner_app)
		AND loan_flag IN ('复贷', '加贷')
		AND loan_status = 'success'
		AND DATE(loan_time) >= '2024-11-01'
		AND DATE(loan_time) <= '2025-11-20'
	)a --每次更新一下日期条件 
	LEFT JOIN xyf_dwd.dwd_risk_model_b_card_df b
	ON a.cust_no = b.cust_no AND b.pt >= '20240901' AND DATE(a.first_order_time) = DATE(b.decision_time)
)
SELECT  substr(loan_time,1,7)              AS loan_mth
       ,loan_model_score_category
       ,COUNT(DISTINCT first_order_number) AS order_cnt
       ,SUM(loan_amt)                      AS loan_amt
FROM loan_risk_score
GROUP BY  substr(loan_time,1,7)
         ,loan_model_score_category;

--========================
--14 客群池量级
--========================
WITH market_portfolio AS
(
	SELECT  *
	       ,CASE WHEN apply_active_last_apply = '高_day30' AND login_active_last_login = '高_day30' THEN '高_day30'
	             WHEN apply_active_last_apply = '中_day30_90' AND login_active_last_login = '中_day30_90' THEN '中_day30_90'
	             WHEN apply_active_last_apply = '高_day30' AND login_active_last_login = '中_day30_90' THEN '中_day30_90'
	             WHEN apply_active_last_apply = '中_day30_90' AND login_active_last_login = '高_day30' THEN '中_day30_90'
	             WHEN apply_active_last_apply = '低_day90_360' AND login_active_last_login = '低_day90_360' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '高_day30' AND login_active_last_login = '低_day90_360' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '中_day30_90' AND login_active_last_login = '低_day90_360' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '低_day90_360' AND login_active_last_login = '中_day30_90' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '低_day90_360' AND login_active_last_login = '高_day30' THEN '低_day90_360'  ELSE '睡眠_day360+' END AS user_tag
	FROM
	(
		SELECT  *
		       ,CASE WHEN customer_pool IN ('API首贷池','API复贷池') THEN 'API'
		             WHEN customer_pool IN ('APP首贷池','APP复贷池') THEN 'APP' END                                                                            AS api_app_pool
		       ,CASE WHEN regulation_reason IN ('不管制' ) THEN '可发标'
		             WHEN regulation_reason IN ('可用额度低于500','禁申','可用额度低于1k') THEN '可经营不可发标'
		             WHEN regulation_reason NOT IN ('不管制' ,'可用额度低于500','禁申') THEN '不可经营' END                                                             AS regulation_type
		       ,CASE WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) <= 30 THEN '高_day30'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) BETWEEN 30 AND 90 THEN '中_day30_90'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) BETWEEN 90 AND 360 THEN '低_day90_360'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) > 360 THEN '睡眠_day360+' END AS apply_active_last_apply
		       ,CASE WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) <= 30 THEN '高_day30'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) BETWEEN 30 AND 90 THEN '中_day30_90'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) BETWEEN 90 AND 360 THEN '低_day90_360'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) > 360 THEN '睡眠_day360+' END AS login_active_last_login
		FROM xyf_ads.ads_user_market_portfolio_label_df
		WHERE (pt IN ( SELECT DISTINCT day_id FROM xyf_dim.dim_pub_date WHERE day_id = day_monthend) OR pt = '20251120')
		AND pt >= '20240901'
		AND customer_pool IN ('API首贷池', 'API复贷池', 'APP首贷池', 'APP复贷池') 
	)
)
SELECT  pt
       ,regulation_type
       ,COUNT(DISTINCT user_no) AS user_no_cnt
FROM market_portfolio
GROUP BY  pt
         ,regulation_type
ORDER BY  pt
         ,regulation_type;

--========================
--15 客群池转化
--========================
WITH market_portfolio AS
(
	SELECT  *
	       ,CASE WHEN apply_active_last_apply = '高_day30' AND login_active_last_login = '高_day30' THEN '高_day30'
	             WHEN apply_active_last_apply = '中_day30_90' AND login_active_last_login = '中_day30_90' THEN '中_day30_90'
	             WHEN apply_active_last_apply = '高_day30' AND login_active_last_login = '中_day30_90' THEN '中_day30_90'
	             WHEN apply_active_last_apply = '中_day30_90' AND login_active_last_login = '高_day30' THEN '中_day30_90'
	             WHEN apply_active_last_apply = '低_day90_360' AND login_active_last_login = '低_day90_360' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '高_day30' AND login_active_last_login = '低_day90_360' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '中_day30_90' AND login_active_last_login = '低_day90_360' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '低_day90_360' AND login_active_last_login = '中_day30_90' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '低_day90_360' AND login_active_last_login = '高_day30' THEN '低_day90_360'  ELSE '睡眠_day360+' END AS user_tag
	FROM
	(
		SELECT  *
		       ,CASE WHEN customer_pool IN ('API首贷池','API复贷池') THEN 'API'
		             WHEN customer_pool IN ('APP首贷池','APP复贷池') THEN 'APP' END                                                                            AS api_app_pool
		       ,CASE WHEN regulation_reason IN ('不管制' ) THEN '可发标'
		             WHEN regulation_reason IN ('可用额度低于500','禁申') THEN '可经营不可发标'
		             WHEN regulation_reason NOT IN ('不管制' ,'可用额度低于500','禁申','可用额度低于1k') THEN '不可经营' END                                                  AS regulation_type
		       ,CASE WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) <= 30 THEN '高_day30'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) BETWEEN 30 AND 90 THEN '中_day30_90'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) BETWEEN 90 AND 360 THEN '低_day90_360'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) > 360 THEN '睡眠_day360+' END AS apply_active_last_apply
		       ,CASE WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) <= 30 THEN '高_day30'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) BETWEEN 30 AND 90 THEN '中_day30_90'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) BETWEEN 90 AND 360 THEN '低_day90_360'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) > 360 THEN '睡眠_day360+' END AS login_active_last_login
		FROM xyf_ads.ads_user_market_portfolio_label_df
		WHERE (pt IN ( SELECT DISTINCT day_id FROM xyf_dim.dim_pub_date WHERE day_id = day_month01) OR pt = '20251101')
		AND pt >= '20240901'
		AND customer_pool IN ('API首贷池', 'API复贷池', 'APP首贷池', 'APP复贷池') 
	)
)
SELECT  a.pt
       ,a.regulation_type
       ,a.user_tag
       ,COUNT(DISTINCT user_no) user_cnt
       ,COUNT(DISTINCT b.app_user_id) applied_user_cnt
       ,COUNT(DISTINCT CASE WHEN b.risk_status = 'pass' THEN b.app_user_id END) risk_passed_user_cnt
       ,COUNT(DISTINCT CASE WHEN b.loan_status = 'success' THEN b.app_user_id END) loan_succeed_user_cnt
       ,SUM(CASE WHEN b.loan_status = 'success' THEN b.amount END) loan_succeed_amount
FROM market_portfolio a
LEFT JOIN xyf_dws.dws_inloan_user_order_hf_v b
ON a.user_no = b.app_user_id AND DATE(b.first_order_time) > DATE(TO_DATE(a.pt, 'yyyymmdd') ) 
AND DATE(b.first_order_time) < DATE(date_add(TO_DATE(a.pt, 'yyyymmdd'), 30)) 
AND b.loan_type_flag IN ('加贷', '复贷') AND b.inner_app = b.app AND app IN ('xyf01', 'fxk')
GROUP BY  a.pt
         ,a.regulation_type
         ,a.user_tag;

--========================
--10 可经营人群
--========================

WITH market_portfolio AS
(
	SELECT  *
	       ,CASE WHEN apply_active_last_apply = '高_day30' AND login_active_last_login = '高_day30' THEN '高_day30'
	             WHEN apply_active_last_apply = '中_day30_90' AND login_active_last_login = '中_day30_90' THEN '中_day30_90'
	             WHEN apply_active_last_apply = '高_day30' AND login_active_last_login = '中_day30_90' THEN '中_day30_90'
	             WHEN apply_active_last_apply = '中_day30_90' AND login_active_last_login = '高_day30' THEN '中_day30_90'
	             WHEN apply_active_last_apply = '低_day90_360' AND login_active_last_login = '低_day90_360' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '高_day30' AND login_active_last_login = '低_day90_360' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '中_day30_90' AND login_active_last_login = '低_day90_360' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '低_day90_360' AND login_active_last_login = '中_day30_90' THEN '低_day90_360'
	             WHEN apply_active_last_apply = '低_day90_360' AND login_active_last_login = '高_day30' THEN '低_day90_360'  ELSE '睡眠_day360+' END AS user_tag
	FROM
	(
		SELECT  *
		       ,CASE WHEN customer_pool IN ('API首贷池','API复贷池') THEN 'API'
		             WHEN customer_pool IN ('APP首贷池','APP复贷池') THEN 'APP' END                                                                            AS api_app_pool
		       ,CASE WHEN regulation_reason IN ('不管制' ) THEN '可发标'
		             WHEN regulation_reason IN ('可用额度低于500','禁申','可用额度低于1k') THEN '可经营不可发标'
		             WHEN regulation_reason NOT IN ('不管制' ,'可用额度低于500','禁申') THEN '不可经营' END                                                             AS regulation_type
		       ,CASE WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) <= 30 THEN '高_day30'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) BETWEEN 30 AND 90 THEN '中_day30_90'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) BETWEEN 90 AND 360 THEN '低_day90_360'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_apply_time)) > 360 THEN '睡眠_day360+' END AS apply_active_last_apply
		       ,CASE WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) <= 30 THEN '高_day30'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) BETWEEN 30 AND 90 THEN '中_day30_90'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) BETWEEN 90 AND 360 THEN '低_day90_360'
		             WHEN DATEDIFF(DATE(CONCAT(SUBSTR(pt,1,4),'-',SUBSTR(pt,5,2),'-',SUBSTR(pt,7,2))),DATE(last_login_time)) > 360 THEN '睡眠_day360+' END AS login_active_last_login
		FROM xyf_ads.ads_user_market_portfolio_label_df
		WHERE (pt IN ( SELECT DISTINCT day_id FROM xyf_dim.dim_pub_date WHERE day_id = day_monthend) OR pt = '20251120')
		AND pt >= '20240601'
		AND customer_pool IN ('API首贷池', 'API复贷池', 'APP首贷池', 'APP复贷池')
	)
)
SELECT  pt
       ,user_tag
       ,COUNT(DISTINCT user_no) AS user_no_cnt
FROM market_portfolio
WHERE regulation_type IN ('可发标', '可经营不可发标')
GROUP BY  pt
         ,user_tag;

